In [ ]:
!pip install -q -U transformers accelerate requests

import os
import re
import json
import glob
import torch
from collections import Counter
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=256 * 28 * 28,
    max_pixels=1024 * 28 * 28,
)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "12GiB", 1: "12GiB"},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

ln_f = model.model.language_model.norm
lm_head = model.lm_head
layers = model.model.language_model.layers

n_layer = model.config.text_config.num_hidden_layers
n_head = model.config.text_config.num_attention_heads
hidden_dim = model.config.text_config.hidden_size
head_dim = layers[0].self_attn.o_proj.in_features // n_head
rms_eps = model.config.text_config.rms_norm_eps
assert len(layers) == n_layer
print(n_layer, "language model layers,", n_head, "heads,", head_dim, "head dim,", model.config.text_config.num_key_value_heads, "kv heads")

In [ ]:
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/
for zpath in ["/kaggle/working/VG_100K.zip", "/kaggle/working/VG_100K_2.zip"]:
    if os.path.exists(zpath):
        os.remove(zpath)
print("Visual Genome downloaded, extracted, zips removed")

In [ ]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]


def load_image(path):
    return Image.open(path).convert("RGB")


def build_inputs(image, text):
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt_text = processor.apply_chat_template(conversation, add_generation_prompt=True)
    return processor(text=prompt_text, images=image, return_tensors="pt").to(model.device, torch.float16)


def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None


def resolve_image_path(q: dict) -> str:
    qid = q["question_id"]
    if qid.startswith("amber_"):
        return os.path.join(AMBER_IMAGE_DIR, f"{q['image_id']}.jpg")
    elif qid.startswith("reefknot_"):
        return find_vg_image_path(q["image_id"])
    return None

In [ ]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"

with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)

assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}
print(len(all_questions), "questions loaded")

In [ ]:
def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


def cache_files(t):
    files = sorted(
        glob.glob(f"/kaggle/input/datasets/*/question-aware-cache-qwen25vl-{t}/{t}_batch_*.pt"),
        key=batch_number,
    )
    assert files, f"attach question-aware-cache-qwen25vl-{t}"
    return files


SOURCES = [
    ("attribute/amber", "attribute", "amber_"),
    ("relation/amber", "relation", "amber_"),
    ("relation/reefknot", "relation", "reefknot_"),
]
N_PER_SOURCE = 16
YESNO = ("yes", "no")

answer_ids = {}
top_counts = {}
samples = {}
for label, t, prefix in SOURCES:
    picked = []
    top_counts[label] = Counter()
    for path in cache_files(t):
        batch = torch.load(path, mmap=True)
        for e in batch:
            if not e["question_id"].startswith(prefix):
                continue
            top_id = int(e["answer_logits"].argmax())
            top_counts[label][top_id] += 1
            answer = e["answer_text"].strip().lower()
            if answer in YESNO:
                answer_ids.setdefault(prefix, {"yes": [], "no": []})[answer].append(top_id)
            if len(picked) < N_PER_SOURCE:
                picked.append({
                    "question_id": e["question_id"],
                    "prefix": prefix,
                    "answer_text": e["answer_text"],
                    "top_id": top_id,
                    "answer_logits": e["answer_logits"].clone(),
                })
        del batch
        if len(picked) == N_PER_SOURCE:
            break
    samples[label] = picked

PAIRS = {}
for prefix, d in answer_ids.items():
    yes_id = Counter(d["yes"]).most_common(1)[0][0]
    no_id = Counter(d["no"]).most_common(1)[0][0]
    PAIRS[prefix] = (yes_id, no_id)
    yes_purity = d["yes"].count(yes_id) / len(d["yes"])
    no_purity = d["no"].count(no_id) / len(d["no"])
    print(f"{prefix:10s} yes_id={yes_id} {processor.tokenizer.decode(yes_id)!r} purity={yes_purity:.4f} | "
          f"no_id={no_id} {processor.tokenizer.decode(no_id)!r} purity={no_purity:.4f}")
    assert yes_purity > 0.9 and no_purity > 0.9, f"mixed casing for {prefix}"
assert set(PAIRS) == {"amber_", "reefknot_"}, set(PAIRS)

for label, _, prefix in SOURCES:
    total = sum(top_counts[label].values())
    in_pair = sum(top_counts[label][i] for i in PAIRS[prefix])
    print(f"{label:18s} top-1 in the yes/no pair: {in_pair}/{total} ({in_pair / total:.4f}) over the scanned batches")

assert all(len(v) == N_PER_SOURCE for v in samples.values()), {k: len(v) for k, v in samples.items()}
for entries in samples.values():
    for e in entries:
        image_path = resolve_image_path(question_by_id[e["question_id"]])
        assert image_path is not None and os.path.exists(image_path), f"missing image for {e['question_id']}"
print({k: len(v) for k, v in samples.items()})

In [ ]:
_state = {"prefix": None}
_u_cache = {}

U = {
    prefix: ln_f.weight.float().cpu() * (lm_head.weight[yes_id] - lm_head.weight[no_id]).float().cpu()
    for prefix, (yes_id, no_id) in PAIRS.items()
}


def u_on(device):
    key = (_state["prefix"], device)
    if key not in _u_cache:
        _u_cache[key] = U[_state["prefix"]].to(device)
    return _u_cache[key]


HEAD_PROJ = {}
with torch.no_grad():
    for prefix in PAIRS:
        _state["prefix"] = prefix
        HEAD_PROJ[prefix] = []
        for layer in layers:
            w = layer.self_attn.o_proj.weight
            HEAD_PROJ[prefix].append(
                w.float().T.contiguous().view(n_head, head_dim, hidden_dim) @ u_on(w.device)
            )

_dla_cache = {}


def _embed_hook(module, args, kwargs):
    h = args[0] if args else kwargs["hidden_states"]
    _dla_cache["embed"] = h[0, -1].float() @ u_on(h.device)


def _make_attn_hook(l):
    def hook(module, args):
        z = args[0][0, -1].float().reshape(n_head, head_dim)
        _dla_cache[f"attn_{l}"] = (z * HEAD_PROJ[_state["prefix"]][l]).sum(-1)
    return hook


def _make_mlp_hook(l):
    def hook(module, args, output):
        _dla_cache[f"mlp_{l}"] = output[0, -1].float() @ u_on(output.device)
    return hook


def _final_norm_hook(module, args):
    h = args[0][0, -1].float()
    _dla_cache["resid_dot"] = h @ u_on(h.device)
    _dla_cache["rms"] = torch.sqrt(h.pow(2).mean() + rms_eps)


_hook_handles = [layers[0].register_forward_pre_hook(_embed_hook, with_kwargs=True)]
for l, layer in enumerate(layers):
    _hook_handles.append(layer.self_attn.o_proj.register_forward_pre_hook(_make_attn_hook(l)))
    _hook_handles.append(layer.mlp.register_forward_hook(_make_mlp_hook(l)))
_hook_handles.append(ln_f.register_forward_pre_hook(_final_norm_hook))
print(len(_hook_handles), "hooks registered")

In [ ]:
@torch.no_grad()
def dla_question(image_path, question_text, prefix):
    _state["prefix"] = prefix
    yes_id, no_id = PAIRS[prefix]
    image = load_image(image_path)
    inputs = build_inputs(image, text=question_text)

    _dla_cache.clear()
    outputs = model(**inputs)
    last_logits = outputs.logits[0, -1].float().cpu()

    return {
        "embed": _dla_cache["embed"].cpu(),
        "attn": torch.stack([_dla_cache[f"attn_{l}"].cpu() for l in range(n_layer)]),
        "mlp": torch.stack([_dla_cache[f"mlp_{l}"].cpu() for l in range(n_layer)]),
        "resid_dot": _dla_cache["resid_dot"].cpu(),
        "rms": _dla_cache["rms"].cpu(),
        "logit_diff": last_logits[yes_id] - last_logits[no_id],
        "top_id": int(last_logits.argmax()),
        "answer_text": processor.tokenizer.decode(last_logits.argmax().item()).strip(),
    }

In [ ]:
results = {label: [] for label in samples}
rows = []

for label, entries in samples.items():
    for e in entries:
        q = question_by_id[e["question_id"]]
        yes_id, no_id = PAIRS[e["prefix"]]
        r = dla_question(resolve_image_path(q), q["question_text"], e["prefix"])
        total = r["embed"] + r["attn"].sum() + r["mlp"].sum()
        scale = r["embed"].abs() + r["attn"].abs().sum() + r["mlp"].abs().sum()
        cached_diff = e["answer_logits"][yes_id].float() - e["answer_logits"][no_id].float()
        if e["top_id"] == yes_id:
            sign_ok = bool(r["logit_diff"] > 0)
        elif e["top_id"] == no_id:
            sign_ok = bool(r["logit_diff"] < 0)
        else:
            sign_ok = True
        rows.append({
            "source": label,
            "rel_err_resid": ((total - r["resid_dot"]).abs() / scale).item(),
            "err_logit": (total / r["rms"] - r["logit_diff"]).abs().item(),
            "err_cache": (r["logit_diff"] - cached_diff).abs().item(),
            "answer_match": r["answer_text"] == e["answer_text"],
            "sign_ok": sign_ok,
            "in_pair": e["top_id"] in (yes_id, no_id),
            "finite": bool(torch.isfinite(total) and torch.isfinite(r["logit_diff"])),
        })
        results[label].append(r)

print(len(rows), "questions run")

In [ ]:
n = len(rows)
for label in samples:
    sub = [r for r in rows if r["source"] == label]
    print(f"{label:18s} rel err resid {max(r['rel_err_resid'] for r in sub):.5f} | "
          f"err logit {max(r['err_logit'] for r in sub):.4f} | err cache {max(r['err_cache'] for r in sub):.4f} | "
          f"answers match {sum(r['answer_match'] for r in sub)}/{len(sub)} | "
          f"sign ok {sum(r['sign_ok'] for r in sub)}/{len(sub)} | top-1 in pair {sum(r['in_pair'] for r in sub)}/{len(sub)}")

assert all(r["finite"] for r in rows), "non-finite values"
assert max(r["rel_err_resid"] for r in rows) < 0.05, "components do not sum to the final residual"
assert max(r["err_logit"] for r in rows) < 0.5, "components do not reproduce the logit difference"
assert max(r["err_cache"] for r in rows) < 0.25, "prompt or token ids differ from the question-aware cache"
assert sum(r["answer_match"] for r in rows) == n, "answers differ from the question-aware cache"
assert all(r["sign_ok"] for r in rows), "logit difference sign disagrees with the cached answer"
print("validation passed")

In [ ]:
for label, rs in results.items():
    attn = torch.stack([r["attn"] / r["rms"] for r in rs]).mean(0).flatten()
    mlp = torch.stack([r["mlp"] / r["rms"] for r in rs]).mean(0)
    mean_diff = torch.stack([r["logit_diff"] for r in rs]).mean().item()
    print(f"--- {label}: mean logit diff {mean_diff:.3f} ---")
    for i in attn.abs().topk(5).indices.tolist():
        print(f"  layer {i // n_head:2d} ({(i // n_head) / n_layer:.0%} depth) head {i % n_head:2d}  {attn[i]:+.3f}")
    for i in mlp.abs().topk(3).indices.tolist():
        print(f"  layer {i:2d} ({i / n_layer:.0%} depth) mlp      {mlp[i]:+.3f}")